In [0]:
# =============================================================================
# FILE:    silver_config.py
# PURPOSE: Single source of truth for ALL Silver layer configuration.
#          Every Silver notebook imports from here. Nothing is hardcoded
#          inside individual notebooks.
#
# DESIGN PRINCIPLE:
#   If a value might change (path, threshold, merge key, column name),
#   it belongs here — not inside a notebook. This means:
#     - Changing the ADLS account name → edit one line here, all notebooks update
#     - Changing a dedup key → edit one entry here, no risk of inconsistency
#     - Onboarding a new team member → they read this file to understand the layer
#
# HOW TO USE IN A NOTEBOOK:
#   # At the top of every Silver notebook:
#   %run ./silver_config
#   # Then use constants directly:
#   df = spark.read.format("delta").load(SilverPaths.BRONZE_COINS_MARKETS)
#
# SECTIONS:
#   1. Account & Container Roots
#   2. Bronze Input Paths
#   3. Silver Output Paths
#   4. Logging Paths
#   5. Checkpoint & Schema Paths  (Autoloader state — Bronze layer, referenced here for Silver reads)
#   6. Delta Merge Keys           (primary key combinations used in MERGE upserts)
#   7. Data Quality Thresholds    (validation rules: min price, max nulls, etc.)
#   8. Silver Schema Column Lists (exact column order expected after transformation)
#   9. Pipeline Run Settings

In [0]:
import uuid
from datetime import datetime, timezone
 
 
# =============================================================================
# SECTION 1 — ACCOUNT & CONTAINER ROOTS
# NOTE: ADLS_ACCOUNT_NAME is read from Key Vault at runtime.
#       We define a helper that builds path roots once the account name is known.
#       Call SilverConfig.init(adls_name) at the top of each notebook.
# =============================================================================
 
class SilverConfig:
    """
    Runtime-initialised config. Call SilverConfig.init(adls_name) once
    per notebook after reading the secret from Key Vault, then access all
    paths and settings as class attributes.
    """
 
    # ── Set by init() ─────────────────────────────────────────────────────────
    ADLS_ACCOUNT_NAME : str = ""
 
    # Container roots (populated by init)
    LANDING_ROOT  : str = ""
    BRONZE_ROOT   : str = ""
    SILVER_ROOT   : str = ""
    LOGGING_ROOT  : str = ""
 
    # Run-level metadata (set once per notebook execution)
    RUN_ID    : str = ""
    RUN_TS    : datetime = None
    DATE_PATH : str = ""   # YYYY/MM/DD  — used in log file paths
 
    @classmethod
    def init(cls, adls_account_name: str) -> None:
        """
        Initialise all path constants from the ADLS account name.
        Must be called once at the top of every Silver notebook before
        using any path or config constant.
 
        Args:
            adls_account_name: Storage account name from Key Vault
                               (e.g., "stcryptocapstone")
        """
        cls.ADLS_ACCOUNT_NAME = adls_account_name
 
        # abfss://  = Azure Blob File System Secure (required for ADLS Gen2)
        # Each container is a separate abfss:// root
        cls.LANDING_ROOT = f"abfss://capstone@{adls_account_name}.dfs.core.windows.net"
        cls.BRONZE_ROOT  = f"abfss://capstone@{adls_account_name}.dfs.core.windows.net/bronze"
        cls.SILVER_ROOT  = f"abfss://capstone@{adls_account_name}.dfs.core.windows.net/silver"
        cls.LOGGING_ROOT = f"abfss://capstone@{adls_account_name}.dfs.core.windows.net/logs"
 
        # Run-level metadata
        cls.RUN_ID    = str(uuid.uuid4())
        cls.RUN_TS    = datetime.now(timezone.utc)
        cls.DATE_PATH = cls.RUN_TS.strftime("%Y/%m/%d")
 

In [0]:
# =============================================================================
# SECTION 2 — BRONZE INPUT PATHS
# These are the Delta table paths written by 02_bronze_autoloader.py.
# Silver notebooks READ from these paths — never write back to them.
# =============================================================================
 
class BronzePaths:
    """
    Full ADLS paths for the five Bronze Delta tables.
    Populated after SilverConfig.init() is called.
 
    Usage:
        BronzePaths.init(SilverConfig.BRONZE_ROOT)
        df = spark.read.format("delta").load(BronzePaths.COINS_MARKETS)
    """
    COINS_MARKETS : str = ""
    OHLC          : str = ""
    MARKET_CHART  : str = ""
    TRENDING      : str = ""
    GLOBAL        : str = ""
 
    @classmethod
    def init(cls, bronze_root: str) -> None:
        cls.COINS_MARKETS = f"{bronze_root}/coins_markets_raw"
        cls.OHLC          = f"{bronze_root}/ohlc_raw"
        cls.MARKET_CHART  = f"{bronze_root}/market_chart_raw"
        cls.TRENDING      = f"{bronze_root}/trending_raw"
        cls.GLOBAL        = f"{bronze_root}/global_raw"
 
 

In [0]:
# =============================================================================
# SECTION 3 — SILVER OUTPUT PATHS
# These are the Delta table paths that Silver notebooks write to.
# Gold notebooks will READ from these paths.
# =============================================================================
 
class SilverPaths:
    """
    Full ADLS paths for the five Silver Delta tables.
 
    Usage:
        SilverPaths.init(SilverConfig.SILVER_ROOT)
        df.write.format("delta").save(SilverPaths.MARKET_SNAPSHOT)
    """
    MARKET_SNAPSHOT    : str = ""
    OHLC_HISTORY       : str = ""
    HOURLY_TIMESERIES  : str = ""
    TRENDING_COINS     : str = ""
    GLOBAL_STATS       : str = ""
 
    @classmethod
    def init(cls, silver_root: str) -> None:
        cls.MARKET_SNAPSHOT   = f"{silver_root}/market_snapshot"
        cls.OHLC_HISTORY      = f"{silver_root}/ohlc_history"
        cls.HOURLY_TIMESERIES = f"{silver_root}/hourly_timeseries"
        cls.TRENDING_COINS    = f"{silver_root}/trending_coins"
        cls.GLOBAL_STATS      = f"{silver_root}/global_stats"
 
 

In [0]:
# =============================================================================
# SECTION 3b — WATERMARK TABLE PATH
#
# WHY A WATERMARK TABLE?
#   Instead of scanning ALL Bronze rows with MAX(bronze_loaded_at) every run
#   (which grows more expensive as Bronze accumulates), we maintain a tiny
#   Delta table with one row per Bronze source table, recording the last
#   bronze_loaded_at timestamp that Silver successfully processed.
#
#   On each Silver run:
#     1. READ  watermark → get last_processed_ts for this Bronze table
#     2. FILTER Bronze   → bronze_loaded_at > last_processed_ts  (only new rows)
#     3. Transform + MERGE into Silver
#     4. UPDATE watermark → set last_processed_ts = MAX(bronze_loaded_at) of this batch
#
#   This means Bronze is never fully scanned. Delta's _delta_log + Z-ORDER
#   on bronze_loaded_at means only the relevant new files are opened.
#
# TABLE SCHEMA (one row per Bronze source table):
#   source_table       STRING  — e.g., "coins_markets_raw"
#   last_processed_ts  TIMESTAMP — latest bronze_loaded_at fully processed by Silver
#   updated_at         TIMESTAMP — when this watermark row was last updated
# =============================================================================
 
class WatermarkPaths:
    """
    Path for the shared Silver watermark Delta table.
    One table, one row per Bronze source — tracks incremental progress.
    """
    WATERMARK_TABLE : str = ""
 
    # Logical names used as the source_table key in the watermark table.
    # These must exactly match the strings passed to get/update watermark calls.
    COINS_MARKETS  = "coins_markets_raw"
    OHLC           = "ohlc_raw"
    MARKET_CHART   = "market_chart_raw"
    TRENDING       = "trending_raw"
    GLOBAL         = "global_raw"
 
    @classmethod
    def init(cls, silver_root: str) -> None:
        # Lives in the silver container — it's Silver's internal state table,
        # not a Bronze or logging concern.
        cls.WATERMARK_TABLE = f"{silver_root}/_watermarks/bronze_watermarks"
 


In [0]:
# =============================================================================
# SECTION 4 — LOGGING PATHS
# One JSON log file per notebook per day, written to the logging container.
# Sub-divided by layer (bronze / silver / gold) so logs are easy to find.
# =============================================================================
 
class LogPaths:
    """
    ADLS log file paths for Silver layer notebooks.
    One JSON file per notebook run, date-partitioned.
 
    Usage:
        LogPaths.init(SilverConfig.LOGGING_ROOT, SilverConfig.DATE_PATH)
        # Then in your notebook:
        SilverUtils.write_run_log(summary_dict, LogPaths.MARKET_SNAPSHOT)
    """
    MARKET_SNAPSHOT   : str = ""
    OHLC_HISTORY      : str = ""
    HOURLY_TIMESERIES : str = ""
    TRENDING_COINS    : str = ""
    GLOBAL_STATS      : str = ""
 
    @classmethod
    def init(cls, logging_root: str, date_path: str, run_id: str) -> None:
        """
        Build log file paths.
        Format: logs/silver/<notebook>/<YYYY>/<MM>/<DD>/<notebook>_<run_id>.json
        """
        base = f"{logging_root}/silver"
        cls.MARKET_SNAPSHOT   = f"{base}/market_snapshot/{date_path}/run_{run_id}.json"
        cls.OHLC_HISTORY      = f"{base}/ohlc_history/{date_path}/run_{run_id}.json"
        cls.HOURLY_TIMESERIES = f"{base}/hourly_timeseries/{date_path}/run_{run_id}.json"
        cls.TRENDING_COINS    = f"{base}/trending_coins/{date_path}/run_{run_id}.json"
        cls.GLOBAL_STATS      = f"{base}/global_stats/{date_path}/run_{run_id}.json"
 

In [0]:
# =============================================================================
# SECTION 5 — DELTA MERGE KEYS
# The column combination that uniquely identifies one row in each Silver table.
# Used in the MERGE (upsert) statement to decide: INSERT new or skip duplicate.
#
# WHY MERGE INSTEAD OF OVERWRITE?
#   If the pipeline re-runs (due to a bug fix or retry), MERGE ensures we don't
#   create duplicate rows. A row already in Silver with the same key is left
#   unchanged; only genuinely new rows are inserted.
# =============================================================================
 
class MergeKeys:
    """
    Primary key column combinations for each Silver Delta table.
    These are used in the MERGE condition:
        "existing.coin_id = new.coin_id AND existing.snapshot_date = new.snapshot_date"
    """
 
    # market_snapshot: one row per coin per day
    MARKET_SNAPSHOT = ["coin_id", "snapshot_date"]
 
    # ohlc_history: one row per OHLC candle per coin per timestamp
    OHLC_HISTORY    = ["coin_id", "ohlc_timestamp"]
 
    # hourly_timeseries: one row per coin per hour
    HOURLY_TIMESERIES = ["coin_id", "hour_timestamp"]
 
    # trending_coins: one row per trending position per day
    TRENDING_COINS  = ["trend_run_date", "coin_id"]
 
    # global_stats: one row per day (global has no coin_id)
    GLOBAL_STATS    = ["stats_date"]

In [0]:
# =============================================================================
# SECTION 6 — DATA QUALITY THRESHOLDS
# Hard rules applied during Silver validation. Any row violating these is
# dropped with a warning logged. Thresholds are here so they can be tuned
# without touching transformation code.
# =============================================================================
 
class DataQuality:
    """
    Validation thresholds for Silver layer data quality checks.
    """
 
    # market_snapshot
    MIN_PRICE_USD          = 0.0        # current_price_usd must be > this
    MIN_MARKET_CAP_USD     = 0          # market_cap_usd must be > this (0 = allow any positive)
    MAX_MARKET_CAP_RANK    = 10_000     # sanity check: no coin should rank > 10,000
    MAX_PRICE_CHANGE_PCT   = 10_000.0   # anything over 10,000% change is likely bad data
 
    # ohlc_history
    # high must be >= low (basic OHLC sanity)
    REQUIRE_HIGH_GTE_LOW   = True
    # close must be >= 0 (no negative prices)
    MIN_OHLC_PRICE         = 0.0
 
    # hourly_timeseries
    MIN_VOLUME_USD         = 0          # volume cannot be negative
 
    # global_stats
    MIN_ACTIVE_CRYPTOS     = 1          # must track at least 1 active crypto
    MIN_BTC_DOMINANCE_PCT  = 0.0        # BTC dominance as % (0–100)
    MAX_BTC_DOMINANCE_PCT  = 100.0
 
    # General
    # If more than this fraction of rows in a batch are dropped by validation,
    # raise a DataQualityError instead of proceeding silently.
    MAX_DROP_FRACTION      = 0.30       # 30% — more than this is suspicious
 

In [0]:
# =============================================================================
# SECTION 7 — SILVER COLUMN LISTS
# Exact final column order for each Silver table after all transformations.
# Used to SELECT and reorder columns before the Delta MERGE write,
# ensuring consistent schema regardless of the order Spark infers them.
# =============================================================================
 
class SilverColumns:
    """
    Final column lists for each Silver table in the exact order they
    should appear in the Delta table. Used in a final .select(*cols) call
    before writing.
    """
 
    MARKET_SNAPSHOT = [
        "coin_id",
        "symbol",
        "name",
        "snapshot_date",
        "snapshot_timestamp",
        "current_price_usd",
        "market_cap_usd",
        "market_cap_rank",
        "total_volume_24h_usd",
        "high_24h_usd",
        "low_24h_usd",
        "price_change_24h_usd",
        "price_change_pct_24h",
        "circulating_supply",
        "total_supply",
        "ath_usd",
        "ath_date",
        "pipeline_run_id",
        "silver_processed_timestamp",
    ]
 
    OHLC_HISTORY = [
        "coin_id",
        "ohlc_timestamp",
        "ohlc_date",
        "open_price",
        "high_price",
        "low_price",
        "close_price",
        "pipeline_run_id",
        "silver_processed_timestamp",
    ]
 
    HOURLY_TIMESERIES = [
        "coin_id",
        "hour_timestamp",
        "price_usd",
        "market_cap_usd",
        "volume_usd",
        "silver_processed_timestamp",
    ]
 
    TRENDING_COINS = [
        "trend_run_date",
        "trend_position",
        "coin_id",
        "coin_name",
        "coin_symbol",
        "market_cap_rank",
        "pipeline_run_id",
        "silver_processed_timestamp",
    ]
 
    GLOBAL_STATS = [
        "stats_date",
        "stats_timestamp",
        "total_market_cap_usd",
        "total_volume_24h_usd",
        "btc_dominance_pct",
        "eth_dominance_pct",
        "active_cryptos",
        "market_cap_change_pct_24h",
        "pipeline_run_id",
        "silver_processed_timestamp",
    ]
 

In [0]:
# =============================================================================
# SECTION 8 — PIPELINE SETTINGS
# Databricks Secret Scope name and Key Vault secret keys.
# Centralised here so if you rename a secret, it's one edit.
# =============================================================================
 
class SecretConfig:
    SCOPE             = "keyvaulthp11"
    KEY_ADLS_NAME     = "adls-key"
 
# =============================================================================
# SECTION 9 — CONVENIENCE INIT FUNCTION
# Call this single function at the top of every Silver notebook.
# It initialises all config classes in the right order.
# =============================================================================
 
def init_silver_config(adls_account_name: str) -> None:
    """
    One-call initialiser for all Silver config classes.
    Call this immediately after reading the ADLS account name from Key Vault.
 
    Args:
        adls_account_name: From dbutils.secrets.get(scope, key)
 
    Example (in any Silver notebook):
        adls_name = dbutils.secrets.get(scope=SecretConfig.SCOPE,
                                        key=SecretConfig.KEY_ADLS_NAME)
        init_silver_config(adls_name)
        # Now all path classes are ready:
        df = spark.read.format("delta").load(BronzePaths.COINS_MARKETS)
        df.write.format("delta").save(SilverPaths.MARKET_SNAPSHOT)
    """
    SilverConfig.init(adls_account_name)
    BronzePaths.init(SilverConfig.BRONZE_ROOT)
    SilverPaths.init(SilverConfig.SILVER_ROOT)
    WatermarkPaths.init(SilverConfig.SILVER_ROOT)
    LogPaths.init(
        SilverConfig.LOGGING_ROOT,
        SilverConfig.DATE_PATH,
        SilverConfig.RUN_ID
    )
 